<a href="https://colab.research.google.com/github/SriSharanya-617/positionalembedding/blob/main/positionalembedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

positional encoding+selfattention

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention

input sentence

In [2]:
...

sentence=["I love Deep Learning"]
print(sentence)


['I love Deep Learning']


tokenization

In [4]:
print("Vocabulary:")

vectorizer=TextVectorization(output_mode="int", output_sequence_length=4)
vectorizer.adapt(sentence)

tokens=vectorizer(sentence)
print(vectorizer.get_vocabulary())

print("Tokens:")
print(tokens.numpy())

Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:
[[4 2 5 3]]


word embeddings

In [5]:
embedding_dim=8

embedding_layer=Embedding(input_dim=len(vectorizer.get_vocabulary()),output_dim=embedding_dim)
word_embeddings=embedding_layer(tokens)

print("Word Embeddings:")
print(word_embeddings.numpy())

Word Embeddings:
[[[ 0.03737995  0.00425135  0.0378976  -0.00658443 -0.01681773
   -0.03694134 -0.01090584 -0.02039333]
  [ 0.01143969 -0.04513099 -0.02667888  0.03460615  0.04492759
   -0.01466149  0.04725473  0.01417421]
  [ 0.01568424  0.02503921  0.03921631 -0.02720681  0.04096941
   -0.04816094  0.02811885 -0.04501728]
  [-0.04607046  0.04087469 -0.0103757  -0.04568776 -0.03429866
    0.01843556  0.03717113 -0.04827757]]]


positional encoding

In [7]:
def positional_encoding(max_position,d_model):
    positions=np.arange(max_position) [:,np.newaxis]
    dimensions=np.arange(d_model)[np.newaxis,:]

    angle_rates=1/np.power(10000,(2*(dimensions//2))/np.float32(d_model))

    angle_rads=positions * angle_rates

    PE=np.zeros((max_position,d_model))

    PE[:,0 :: 2]=np.sin(angle_rads[:,0 :: 2])
    PE[:,1 :: 2]=np.cos(angle_rads[:,1 :: 2])

    return tf.cast(PE,dtype=tf.float32)

generate positional encoding

In [8]:
PE = positional_encoding(4, embedding_dim)

print(PE.numpy())

[[ 0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00
   0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00]
 [ 8.4147096e-01  5.4030228e-01  9.9833414e-02  9.9500418e-01
   9.9998331e-03  9.9994999e-01  9.9999981e-04  9.9999952e-01]
 [ 9.0929741e-01 -4.1614684e-01  1.9866933e-01  9.8006660e-01
   1.9998666e-02  9.9980003e-01  1.9999987e-03  9.9999797e-01]
 [ 1.4112000e-01 -9.8999250e-01  2.9552022e-01  9.5533651e-01
   2.9995501e-02  9.9955004e-01  2.9999956e-03  9.9999553e-01]]


Add Positional Encoding

In [9]:
position_aware_embeddings = word_embeddings + PE[tf.newaxis, :]

print("Position-aware Embeddings:")
print(position_aware_embeddings.numpy())

Position-aware Embeddings:
[[[ 0.03737995  1.0042514   0.0378976   0.9934156  -0.01681773
    0.96305865 -0.01090584  0.9796067 ]
  [ 0.85291064  0.49517128  0.07315453  1.0296103   0.05492742
    0.9852885   0.04825473  1.0141737 ]
  [ 0.92498165 -0.39110765  0.23788564  0.95285976  0.06096807
    0.9516391   0.03011885  0.9549807 ]
  [ 0.09504955 -0.9491178   0.2851445   0.9096488  -0.00430316
    1.0179856   0.04017112  0.951718  ]]]


multi head attention

In [13]:
attention_layer=MultiHeadAttention(
num_heads=2,
key_dim=embedding_dim)

apply self attention

In [15]:
attention_output=attention_layer(
query=position_aware_embeddings,
value=position_aware_embeddings,
key=position_aware_embeddings,

)

print(attention_output.shape)
print("Contextulized Embeddings:")
print(attention_output.numpy())

(1, 4, 8)
Contextulized Embeddings:
[[[-0.04664467 -0.15968612 -0.42836955 -0.291229    0.02527206
   -0.5010594  -0.14928752 -0.14727826]
  [-0.04630017 -0.15914255 -0.42783648 -0.2941074   0.02612996
   -0.50287294 -0.15060213 -0.14955965]
  [-0.04706915 -0.16078842 -0.42795444 -0.2910212   0.02614209
   -0.50153184 -0.15144339 -0.14926535]
  [-0.04710227 -0.16307208 -0.4290505  -0.28456748  0.02567774
   -0.4982276  -0.15112595 -0.14659199]]]
